# DepChainTagger

This notebook shows how to configure `DepChainTagger`, how to define a `PathPattern`, and how to run the tagger on a sample text. It also explains the most important parts of the output so you can adapt the example to your own texts.


## What this notebook covers

1. The constructor parameters exposed by `DepChainTagger`.
2. How to build a simple dependency-chain pattern with `NodeConstraint`, `EdgeConstraint`, and `PathPattern`.
3. How to prepare an `estnltk.Text` object with syntax layers.
4. How to run the tagger and inspect the resulting matches.


In [1]:
import estnltk

from scripts.DepChainTagger import (
    ConditionMode,
    DepChainTagger,
    DepTaggerOrchestrator,
    DirectionMode,
    EdgeConstraint,
    NodeConstraint,
    PathPattern,
    ValueCondition,
)

from estnltk_neural.taggers import StanzaSyntaxTagger

e:\Git_projects\EstNLTK\EstNLTK_DependencyChains\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Tagger parameters

The current `DepChainTagger` constructor focuses on output control and match filtering. The tagger always reads from the `stanza_syntax` and `sentences` layers internally, and it writes a relation layer named `dep_chains` by default.

| Parameter                     | What it controls                                                                        |
| ----------------------------- | --------------------------------------------------------------------------------------- |
| `patterns`                    | A tuple of `PathPattern` objects that define what the tagger should match.              |
| `output_layer`                | Name of the relation layer created by the tagger.                                       |
| `output_attributes`           | Extra attributes written to the output layer. If omitted, the package default is used.  |
| `include_pattern_constraints` | Whether to include the pattern constraints as attributes in the output spans/relations. |
| `sentence_match_dedup_mode`   | Deduplication inside one sentence: `none`, `exact`, or `role_based`.                    |
| `max_matches_per_sentence`    | Maximum number of matches to keep per sentence.                                         |
| `allow_role_node_overlap`     | Whether the same node may fill more than one role in a pattern.                         |
| `global_dedup_mode`           | Deduplication across the whole text: `none`, `exact`, or `role_based`.                  |
| `max_total_matches`           | Upper limit for all matches across the entire text.                                     |

Deduplication modes:

- `none`: No deduplication; all matches are kept.
- `exact`: Matches with identical sets of nodes are considered duplicates; only one is kept.
- `role_based`: Matches that share the same node in the same role are considered duplicates; only one is kept.

Deduplication is applied after all matches are found, so it does not affect the matching process itself. The `max_matches_per_sentence` and `max_total_matches` parameters are applied after deduplication, so they limit the number of matches that are kept in the final output.


## Preparing a sample text

The tagger expects a text object with sentence segmentation and a syntax layer.


In [2]:
# Download the Stanza models for Estonian if not already present
stanza_syntax_tagger = StanzaSyntaxTagger(
    input_type="morph_analysis",
    input_morph_layer="morph_analysis",
    add_parent_and_children=True,
)
# estnltk.download("stanzasyntaxtagger")

In [3]:
# sample_text = "Ta andis lendurist abikaasale oma raamatu. See raamat on väga huvitav."
# sample_text = "Üks ütles, et 1. mail tähistab palju maid töörahvapüha."

In [4]:
sample_text = "1990. aasta kuumal suvel vaatas Bureau Veritas Estline'i omanduseks saanud laeva uuesti üle."
text_obj = estnltk.Text(sample_text)
text_obj.tag_layer(
    "morph_extended"
)  # StanzaSyntaxTagger expects morph layer to be present
stanza_syntax_tagger.tag(text_obj)

Text(text="1990. aasta kuumal suvel vaatas Bureau Veritas Estline'i omanduseks saanud laeva uuesti üle.")

Let's look at the syntax layer to see what the syntax tagger has produced for our sample sentence.


In [5]:
display(text_obj.stanza_syntax)

Layer(name='stanza_syntax', attributes=('id', 'lemma', 'upostag', 'xpostag', 'feats', 'head', 'deprel', 'deps', 'misc', 'parent_span', 'children'), spans=SL[Span('1990.', [{'id': 1, 'lemma': '1990.', 'upostag': 'O', 'xpostag': 'O', 'feats': OrderedDict({'?': '?'}), 'head': 2, 'deprel': 'amod', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': ()}]),
Span('aasta', [{'id': 2, 'lemma': 'aasta', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict({'sg': 'sg', 'g': 'g'}), 'head': 4, 'deprel': 'nmod', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': <class 'tuple'>}]),
Span('kuumal', [{'id': 3, 'lemma': 'kuum', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict({'sg': 'sg', 'ad': 'ad'}), 'head': 4, 'deprel': 'amod', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': ()}]),
Span('suvel', [{'id': 4, 'lemma': 'suvi', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict({'sg': 'sg', 'ad': 'ad'}), 'head': 5, 'deprel': 'obl', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': <class 'tuple'>}]),
Span('vaatas', [{'id': 5, 'lemma': 'vaatama', 'upostag': 'V', 'xpostag': 'V', 'feats': OrderedDict({'s': 's'}), 'head': 0, 'deprel': 'root', 'deps': '_', 'misc': '_', 'parent_span': None, 'children': <class 'tuple'>}]),
Span('Bureau', [{'id': 6, 'lemma': 'Bureau', 'upostag': 'H', 'xpostag': 'H', 'feats': OrderedDict({'sg': 'sg', 'n': 'n'}), 'head': 5, 'deprel': 'nsubj', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': <class 'tuple'>}]),
Span('Veritas', [{'id': 7, 'lemma': 'Veritas', 'upostag': 'H', 'xpostag': 'H', 'feats': OrderedDict({'sg': 'sg', 'n': 'n'}), 'head': 6, 'deprel': 'flat', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': ()}]),
Span("Estline'i", [{'id': 8, 'lemma': 'Estline', 'upostag': 'H', 'xpostag': 'H', 'feats': OrderedDict({'sg': 'sg', 'g': 'g'}), 'head': 6, 'deprel': 'flat', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': ()}]),
Span('omanduseks', [{'id': 9, 'lemma': 'omandus', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict({'sg': 'sg', 'tr': 'tr'}), 'head': 10, 'deprel': 'obj', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': ()}]),
Span('saanud', [{'id': 10, 'lemma': 'saanud', 'upostag': 'A', 'xpostag': 'A', 'feats': OrderedDict({'sg': 'sg', 'n': 'n'}), 'head': 11, 'deprel': 'acl', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': <class 'tuple'>}]),
Span('laeva', [{'id': 11, 'lemma': 'laev', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict({'sg': 'sg', 'p': 'p'}), 'head': 5, 'deprel': 'obj', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': <class 'tuple'>}]),
Span('uuesti', [{'id': 12, 'lemma': 'uuesti', 'upostag': 'D', 'xpostag': 'D', 'feats': OrderedDict(), 'head': 5, 'deprel': 'advmod', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': ()}]),
Span('üle', [{'id': 13, 'lemma': 'üle', 'upostag': 'D', 'xpostag': 'D', 'feats': OrderedDict(), 'head': 5, 'deprel': 'compound:prt', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': ()}]),
Span('.', [{'id': 14, 'lemma': '.', 'upostag': 'Z', 'xpostag': 'Z', 'feats': OrderedDict(), 'head': 5, 'deprel': 'punct', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': ()}])])

## Constructing a tagger with a simple pattern


A tagger is configured by passing the desired parameters to the constructor. The most important parameter is `patterns`, which defines what the tagger will look for in the syntax tree. The patterns are defined using a custom format that allows you to specify conditions on nodes and edges, as well as the roles of the matched nodes in the output relation layer.

A pattern is defined with `PathPattern`, which consists of a sequence of `NodeConstraint` and `EdgeConstraint` objects. Each `NodeConstraint` specifies conditions on the nodes in the syntax tree, while each `EdgeConstraint` specifies conditions on the edges between those nodes. The pattern also defines the roles for each node, which determine how the matched nodes will be labelled in the output relation layer.

Below is a simplified overview of the workflow:

```
(NodeConstraints, EdgeConstraints) --> PathPattern (with roles) --> DepChainTagger --> Output relation layer with matches
```


Let's start with a simple pattern that finds a root and its direct children in the syntax layer. We will define a pattern with two nodes: the first node is the root of the sentence, and the second node is any child of the root. We will label the root as `root` and the child as `first_level_child` in the output relation layer. The edge constraint will specify that they can look only one step away from the root.


In [80]:
root_pattern = PathPattern(
    name="root_and_children",
    node_steps=(
        NodeConstraint(
            role="root",
            attribute_conditions={
                "deprel": ValueCondition(mode=ConditionMode.EXACT, value="root")
            },
        ),
        NodeConstraint(
            role="first_level_child",
        ),
    ),
    edge_steps=(
        EdgeConstraint(
            direction=DirectionMode.DOWN,
            min_hops=1,
            max_hops=1,
        ),
    ),
)

Now, let's instantiate the tagger with this pattern and see how it works on our sample text.


In [81]:
verb_tagger = DepChainTagger(patterns=(root_pattern,))

In [82]:
sample_text = "1990. aasta kuumal suvel vaatas Bureau Veritas Estline'i omanduseks saanud laeva uuesti üle."
text_obj = estnltk.Text(sample_text)
text_obj.tag_layer("morph_extended")
stanza_syntax_tagger.tag(text_obj)

# Run the tagger
verb_tagger.tag(text_obj);

Let's check the Text object after tagging.


In [83]:
display(text_obj)

Text(text="1990. aasta kuumal suvel vaatas Bureau Veritas Estline'i omanduseks saanud laeva uuesti üle.")

There is a new relation layer named `dep_chains` that contains the matches found by the tagger. Each match is a relation with a role `verb` that points to a node in the syntax tree that satisfies the condition of being a verb. The exact number of matches depends on the quality of the syntax parse and the Stanza model version used in your environment.


Let's inspect the first match in more detail to see what information is available.


In [84]:
display(text_obj["dep_chains"])
text_obj["dep_chains"].display()

RelationLayer(name='dep_chains', span_names=('root', 'first_level_child'), attributes=('pattern_name', 'matched_text'), relations=[Relation([NamedSpan(root: 'vaatas'), NamedSpan(first_level_child: 'suvel')], [{'pattern_name': 'root_and_children', 'matched_text': 'vaatas suvel'}]), Relation([NamedSpan(root: 'vaatas'), NamedSpan(first_level_child: 'Bureau')], [{'pattern_name': 'root_and_children', 'matched_text': 'vaatas Bureau'}]), Relation([NamedSpan(root: 'vaatas'), NamedSpan(first_level_child: 'laeva')], [{'pattern_name': 'root_and_children', 'matched_text': 'vaatas laeva'}]), Relation([NamedSpan(root: 'vaatas'), NamedSpan(first_level_child: 'uuesti')], [{'pattern_name': 'root_and_children', 'matched_text': 'vaatas uuesti'}]), Relation([NamedSpan(root: 'vaatas'), NamedSpan(first_level_child: 'üle')], [{'pattern_name': 'root_and_children', 'matched_text': 'vaatas üle'}]), Relation([NamedSpan(root: 'vaatas'), NamedSpan(first_level_child: '.')], [{'pattern_name': 'root_and_children', 'matched_text': 'vaatas .'}])])

1990. aasta kuumal suvel first_level_child(0) vaatas root(0), root(1), root(2), root(3), root(4), root(5) Bureau first_level_child(1) Veritas Estline'i omanduseks saanud laeva first_level_child(2) uuesti first_level_child(3) üle first_level_child(4) . first_level_child(5)

The output layer is a relation layer, so each match is represented as a relation with one or more roles.

- The `root` attribute points to the root's `role` we defined in the pattern.
- The `first_level_child` attribute points to the child node's `role` we defined in the pattern.
- The `pattern_name` attribute indicates which pattern was matched.
- The `matched_text` attribute shows the text covered by the matched nodes.

Universally, the output relation layer is structured as follows:

- First come each role defined in the pattern
- Then come the default attributes: `pattern_name` and `matched_text`
- Finally come all the extra attributes related to the constraints defined in the pattern, with the role name as a prefix (e.g., `verb_upostag` for the `upostag` constraint on the `verb` role). By default, these are not included in the output, but you can enable them by setting `include_pattern_constraints=True` when constructing the tagger.


## Going in-depth on patterns and tagger configuration


### Condition modes


<!-- - **EXACT**: Match when the actual value is exactly equal to the expected value.
- **NEGATION**: Match when the actual value is not equal to the expected value.
- **WILDCARD**: Match any value (expected value is ignored, must be None).
- **MEMBERSHIP**: Match when the actual (scalar) value is in the expected iterable
    of condition values.  The *condition* holds a collection; the *attribute* is scalar.
- **NOT_MEMBERSHIP**: Match when the actual (scalar) value is not in the expected
    iterable of condition values.  This is the logical inverse of MEMBERSHIP.
- **REGEX**: Match when the actual value, converted to text, satisfies the given
    regular-expression pattern. This is intended for flexible substring and pattern
    matching on scalar attributes. -->

Constraints can be defined in different modes, which determine how the tagger evaluates them against the syntax tree. These are defined by the `ConditionMode` enum and contain the following options:

- `EXACT`: The constraint must be satisfied exactly as specified.
  - For example, if a `NodeConstraint` specifies attribute with value like `upostag="V"` in `EXACT` mode, it will only match nodes that **do have** the `upostag` attribute with the value `V`.
- `NEGATION`: The constraint is satisfied when the actual value does not match the expected value. It is the logical inverse of `EXACT`.
  - For example, `upostag="V"` in `NEGATION` mode will match nodes that **do not have** the `upostag` attribute with the value `V`.
- `WILDCARD`: The constraint is satisfied regardless of the actual value. The expected value is ignored and must be set to `None`.
  - For example, `upostag=None` in `WILDCARD` mode will match any node regardless of its `upostag` value.
- `MEMBERSHIP`: The constraint is satisfied when the actual value (which must be scalar) is in the expected iterable of condition values.
  - For example, `upostag=["V", "N"]` in `MEMBERSHIP` mode will match nodes that have an `upostag` value of either `V` or `N`.
- `NOT_MEMBERSHIP`: The constraint is satisfied when the actual value (which must be scalar) is not in the expected iterable of condition values. It is the logical inverse of `MEMBERSHIP`.
  - For example, `upostag=["V", "N"]` in `NOT_MEMBERSHIP` mode will match nodes that have an `upostag` value that is neither `V` nor `N`.
- `REGEX`: The constraint is satisfied when the actual value, converted to text, matches the given regular-expression pattern. This allows for flexible substring and pattern matching on scalar attributes.
  - For example, `upostag="^V.*"` in `REGEX` mode will match nodes whose `upostag` value starts with `V`, such as `V`, `VERB`, etc.

**When to use `WILDCARD` and when to use an empty `NodeConstraint`?**

There is two ways to express that we want to match any node regardless of its attributes: we can use a `NodeConstraint` with `WILDCARD` mode, or we can use an empty `NodeConstraint` with no conditions. Both of these will match any node, but they have different implications for how the tagger processes the pattern.

With empty `NodeConstraint`, the tagger will recognize that there are no conditions to check for that node, so it can skip any attribute checks and directly consider all nodes as potential matches for that role. This is more efficient because it avoids unnecessary checks. However, the output layer will not contain any attributes for that role, since there are no conditions defined.

With `WILDCARD` mode, the tagger will still go through the process of checking the node against the constraint, but it will always succeed because the expected value is ignored. This means that the tagger will treat all nodes as potential matches for that role, but it will also include an attribute in the output layer indicating that this role was filled with a wildcard match. This can be useful for debugging or for cases where you want to explicitly indicate that any node can fill that role.


Let's take the example above and modify the pattern to use different condition modes to see how it affects the matches and the output layer. For the brevity of the example, let's define all the patterns at once.


In [42]:
# Pattern definition using EXACT condition mode
v_pattern_exact = PathPattern(
    name="exact_verb_pattern",
    node_steps=(
        NodeConstraint(
            role="verb",
            attribute_conditions={
                "upostag": ValueCondition(mode=ConditionMode.EXACT, value="V")
            },
        ),
    ),
    edge_steps=(),
)
# Pattern definition using NEGATED condition mode
v_pattern_negated = PathPattern(
    name="negation_verb_pattern",
    node_steps=(
        NodeConstraint(
            role="not_verb",
            attribute_conditions={
                "upostag": ValueCondition(mode=ConditionMode.NEGATION, value="V")
            },
        ),
    ),
    edge_steps=(),
)
# Pattern definition using WILDCARD condition mode
v_pattern_wildcard = PathPattern(
    name="wildcard_verb_pattern",
    node_steps=(
        NodeConstraint(
            role="all",
            attribute_conditions={
                "upostag": ValueCondition(mode=ConditionMode.WILDCARD, value=None)
            },
        ),
    ),
    edge_steps=(),
)
# Pattern definition using MEMBERSHIP condition mode
v_and_s_pattern_membership = PathPattern(
    name="membership_verb_and_noun_pattern",
    node_steps=(
        NodeConstraint(
            role="verb_or_noun",
            attribute_conditions={
                "upostag": ValueCondition(
                    mode=ConditionMode.MEMBERSHIP, value=["V", "S"]
                )
            },
        ),
    ),
    edge_steps=(),
)
# Pattern definition using NOT_MEMBERSHIP condition mode
v_and_s_pattern_not_membership = PathPattern(
    name="not_membership_verb_and_noun_pattern",
    node_steps=(
        NodeConstraint(
            role="not_verb_or_noun",
            attribute_conditions={
                "upostag": ValueCondition(
                    mode=ConditionMode.NOT_MEMBERSHIP, value=["V", "S"]
                )
            },
        ),
    ),
    edge_steps=(),
)
# Pattern definition using REGEX condition mode
v_pattern_regex = PathPattern(
    name="regex_verb_pattern",
    node_steps=(
        NodeConstraint(
            role="starts_with_v",
            attribute_conditions={
                "upostag": ValueCondition(mode=ConditionMode.REGEX, value="^V.*")
            },
        ),
    ),
    edge_steps=(),
)

#### Using `EXACT` mode


First is condition mode `EXACT` that we have already seen. It matches only verbs, and the output layer contains the `verb_upostag` attribute with the value `V`.


In [55]:
# Tagger instantiation
verb_tagger = DepChainTagger(patterns=(v_pattern_exact,))

# Sample text with syntax layer
sample_text = "1990. aasta kuumal suvel vaatas Bureau Veritas Estline'i omanduseks saanud laeva uuesti üle."
text_obj = estnltk.Text(sample_text)
text_obj.tag_layer("morph_extended")
stanza_syntax_tagger.tag(text_obj)

# Run the tagger
verb_tagger.tag(text_obj);

In [56]:
display(text_obj["dep_chains"])
text_obj["dep_chains"].display()

RelationLayer(name='dep_chains', span_names=('verb',), attributes=('pattern_name', 'matched_text'), relations=[Relation([NamedSpan(verb: 'vaatas')], [{'pattern_name': 'exact_verb_pattern', 'matched_text': 'vaatas'}])])

1990. aasta kuumal suvel vaatas verb(0) Bureau Veritas Estline'i omanduseks saanud laeva uuesti üle.

Only the word "vaatas" matches the `EXACT` condition, so we get one match with `verb_upostag` equal to `V`.


#### Using `NEGATION` mode


Second is condition mode `NEGATION`, which matches any node that is not a verb, so it will match all other nodes in the syntax layer.


In [45]:
# Tagger instantiation
verb_tagger = DepChainTagger(patterns=(v_pattern_negated,))

# Sample text with syntax layer
sample_text = "1990. aasta kuumal suvel vaatas Bureau Veritas Estline'i omanduseks saanud laeva uuesti üle."
text_obj = estnltk.Text(sample_text)
text_obj.tag_layer("morph_extended")
stanza_syntax_tagger.tag(text_obj)

# Run the tagger
verb_tagger.tag(text_obj);

In [46]:
display(text_obj["dep_chains"])
text_obj["dep_chains"].display()

RelationLayer(name='dep_chains', span_names=('not_verb',), attributes=('pattern_name', 'matched_text'), relations=[Relation([NamedSpan(not_verb: '1990.')], [{'pattern_name': 'negation_verb_pattern', 'matched_text': '1990.'}]), Relation([NamedSpan(not_verb: 'aasta')], [{'pattern_name': 'negation_verb_pattern', 'matched_text': 'aasta'}]), Relation([NamedSpan(not_verb: 'kuumal')], [{'pattern_name': 'negation_verb_pattern', 'matched_text': 'kuumal'}]), Relation([NamedSpan(not_verb: 'suvel')], [{'pattern_name': 'negation_verb_pattern', 'matched_text': 'suvel'}]), Relation([NamedSpan(not_verb: 'Bureau')], [{'pattern_name': 'negation_verb_pattern', 'matched_text': 'Bureau'}]), Relation([NamedSpan(not_verb: 'Veritas')], [{'pattern_name': 'negation_verb_pattern', 'matched_text': 'Veritas'}]), Relation([NamedSpan(not_verb: "Estline'i")], [{'pattern_name': 'negation_verb_pattern', 'matched_text': "Estline'i"}]), Relation([NamedSpan(not_verb: 'omanduseks')], [{'pattern_name': 'negation_verb_pattern', 'matched_text': 'omanduseks'}]), Relation([NamedSpan(not_verb: 'saanud')], [{'pattern_name': 'negation_verb_pattern', 'matched_text': 'saanud'}]), Relation([NamedSpan(not_verb: 'laeva')], [{'pattern_name': 'negation_verb_pattern', 'matched_text': 'laeva'}]), Relation([NamedSpan(not_verb: 'uuesti')], [{'pattern_name': 'negation_verb_pattern', 'matched_text': 'uuesti'}]), Relation([NamedSpan(not_verb: 'üle')], [{'pattern_name': 'negation_verb_pattern', 'matched_text': 'üle'}]), Relation([NamedSpan(not_verb: '.')], [{'pattern_name': 'negation_verb_pattern', 'matched_text': '.'}])])

1990. not_verb(0) aasta not_verb(1) kuumal not_verb(2) suvel not_verb(3) vaatas Bureau not_verb(4) Veritas not_verb(5) Estline'i not_verb(6) omanduseks not_verb(7) saanud not_verb(8) laeva not_verb(9) uuesti not_verb(10) üle not_verb(11) . not_verb(12)

With `NEGATION` mode, we get matches for all nodes that do not have `upostag` equal to `V`.


#### Using `WILDCARD` mode


Third condition mode is `WILDCARD`, which matches any node regardless of its attributes.


In [47]:
# Tagger instantiation
verb_tagger = DepChainTagger(patterns=(v_pattern_wildcard,))

# Sample text with syntax layer
sample_text = "1990. aasta kuumal suvel vaatas Bureau Veritas Estline'i omanduseks saanud laeva uuesti üle."
text_obj = estnltk.Text(sample_text)
text_obj.tag_layer("morph_extended")
stanza_syntax_tagger.tag(text_obj)

# Run the tagger
verb_tagger.tag(text_obj);

In [48]:
display(text_obj["dep_chains"])
text_obj["dep_chains"].display()

RelationLayer(name='dep_chains', span_names=('all',), attributes=('pattern_name', 'matched_text'), relations=[Relation([NamedSpan(all: '1990.')], [{'pattern_name': 'wildcard_verb_pattern', 'matched_text': '1990.'}]), Relation([NamedSpan(all: 'aasta')], [{'pattern_name': 'wildcard_verb_pattern', 'matched_text': 'aasta'}]), Relation([NamedSpan(all: 'kuumal')], [{'pattern_name': 'wildcard_verb_pattern', 'matched_text': 'kuumal'}]), Relation([NamedSpan(all: 'suvel')], [{'pattern_name': 'wildcard_verb_pattern', 'matched_text': 'suvel'}]), Relation([NamedSpan(all: 'vaatas')], [{'pattern_name': 'wildcard_verb_pattern', 'matched_text': 'vaatas'}]), Relation([NamedSpan(all: 'Bureau')], [{'pattern_name': 'wildcard_verb_pattern', 'matched_text': 'Bureau'}]), Relation([NamedSpan(all: 'Veritas')], [{'pattern_name': 'wildcard_verb_pattern', 'matched_text': 'Veritas'}]), Relation([NamedSpan(all: "Estline'i")], [{'pattern_name': 'wildcard_verb_pattern', 'matched_text': "Estline'i"}]), Relation([NamedSpan(all: 'omanduseks')], [{'pattern_name': 'wildcard_verb_pattern', 'matched_text': 'omanduseks'}]), Relation([NamedSpan(all: 'saanud')], [{'pattern_name': 'wildcard_verb_pattern', 'matched_text': 'saanud'}]), Relation([NamedSpan(all: 'laeva')], [{'pattern_name': 'wildcard_verb_pattern', 'matched_text': 'laeva'}]), Relation([NamedSpan(all: 'uuesti')], [{'pattern_name': 'wildcard_verb_pattern', 'matched_text': 'uuesti'}]), Relation([NamedSpan(all: 'üle')], [{'pattern_name': 'wildcard_verb_pattern', 'matched_text': 'üle'}]), Relation([NamedSpan(all: '.')], [{'pattern_name': 'wildcard_verb_pattern', 'matched_text': '.'}])])

1990. all(0) aasta all(1) kuumal all(2) suvel all(3) vaatas all(4) Bureau all(5) Veritas all(6) Estline'i all(7) omanduseks all(8) saanud all(9) laeva all(10) uuesti all(11) üle all(12) . all(13)

As expected, with `WILDCARD` mode we get matches for all nodes in the syntax layer.


#### Using `MEMBERSHIP` mode


The fourth condition mode is `MEMBERSHIP`, which matches nodes whose `upostag` value is either `V` or `S`. This will match verbs and nouns.


In [57]:
# Tagger instantiation
verb_tagger = DepChainTagger(patterns=(v_and_s_pattern_membership,))

# Sample text with syntax layer
sample_text = "1990. aasta kuumal suvel vaatas Bureau Veritas Estline'i omanduseks saanud laeva uuesti üle."
text_obj = estnltk.Text(sample_text)
text_obj.tag_layer("morph_extended")
stanza_syntax_tagger.tag(text_obj)

# Run the tagger
verb_tagger.tag(text_obj);

In [58]:
display(text_obj["dep_chains"])
text_obj["dep_chains"].display()

RelationLayer(name='dep_chains', span_names=('verb_or_noun',), attributes=('pattern_name', 'matched_text'), relations=[Relation([NamedSpan(verb_or_noun: 'aasta')], [{'pattern_name': 'membership_verb_and_noun_pattern', 'matched_text': 'aasta'}]), Relation([NamedSpan(verb_or_noun: 'kuumal')], [{'pattern_name': 'membership_verb_and_noun_pattern', 'matched_text': 'kuumal'}]), Relation([NamedSpan(verb_or_noun: 'suvel')], [{'pattern_name': 'membership_verb_and_noun_pattern', 'matched_text': 'suvel'}]), Relation([NamedSpan(verb_or_noun: 'vaatas')], [{'pattern_name': 'membership_verb_and_noun_pattern', 'matched_text': 'vaatas'}]), Relation([NamedSpan(verb_or_noun: 'omanduseks')], [{'pattern_name': 'membership_verb_and_noun_pattern', 'matched_text': 'omanduseks'}]), Relation([NamedSpan(verb_or_noun: 'laeva')], [{'pattern_name': 'membership_verb_and_noun_pattern', 'matched_text': 'laeva'}])])

1990. aasta verb_or_noun(0) kuumal verb_or_noun(1) suvel verb_or_noun(2) vaatas verb_or_noun(3) Bureau Veritas Estline'i omanduseks verb_or_noun(4) saanud laeva verb_or_noun(5) uuesti üle.

All nodes with `upostag` equal to `V` or `S` are matched.


#### Using `NOT_MEMBERSHIP` mode


The fourth condition mode is `NOT_MEMBERSHIP`, which matches nodes whose `upostag` value is neither `V` nor `S`. This will match all nodes except verbs and nouns.


In [59]:
# Tagger instantiation
verb_tagger = DepChainTagger(patterns=(v_and_s_pattern_not_membership,))

# Sample text with syntax layer
sample_text = "1990. aasta kuumal suvel vaatas Bureau Veritas Estline'i omanduseks saanud laeva uuesti üle."
text_obj = estnltk.Text(sample_text)
text_obj.tag_layer("morph_extended")
stanza_syntax_tagger.tag(text_obj)

# Run the tagger
verb_tagger.tag(text_obj);

In [60]:
display(text_obj["dep_chains"])
text_obj["dep_chains"].display()

RelationLayer(name='dep_chains', span_names=('not_verb_or_noun',), attributes=('pattern_name', 'matched_text'), relations=[Relation([NamedSpan(not_verb_or_noun: '1990.')], [{'pattern_name': 'not_membership_verb_and_noun_pattern', 'matched_text': '1990.'}]), Relation([NamedSpan(not_verb_or_noun: 'Bureau')], [{'pattern_name': 'not_membership_verb_and_noun_pattern', 'matched_text': 'Bureau'}]), Relation([NamedSpan(not_verb_or_noun: 'Veritas')], [{'pattern_name': 'not_membership_verb_and_noun_pattern', 'matched_text': 'Veritas'}]), Relation([NamedSpan(not_verb_or_noun: "Estline'i")], [{'pattern_name': 'not_membership_verb_and_noun_pattern', 'matched_text': "Estline'i"}]), Relation([NamedSpan(not_verb_or_noun: 'saanud')], [{'pattern_name': 'not_membership_verb_and_noun_pattern', 'matched_text': 'saanud'}]), Relation([NamedSpan(not_verb_or_noun: 'uuesti')], [{'pattern_name': 'not_membership_verb_and_noun_pattern', 'matched_text': 'uuesti'}]), Relation([NamedSpan(not_verb_or_noun: 'üle')], [{'pattern_name': 'not_membership_verb_and_noun_pattern', 'matched_text': 'üle'}]), Relation([NamedSpan(not_verb_or_noun: '.')], [{'pattern_name': 'not_membership_verb_and_noun_pattern', 'matched_text': '.'}])])

1990. not_verb_or_noun(0) aasta kuumal suvel vaatas Bureau not_verb_or_noun(1) Veritas not_verb_or_noun(2) Estline'i not_verb_or_noun(3) omanduseks saanud not_verb_or_noun(4) laeva uuesti not_verb_or_noun(5) üle not_verb_or_noun(6) . not_verb_or_noun(7)

All nodes with `upostag` not equal to `V` or `S` are matched.


#### Using `REGEX` mode


The final condition mode is `REGEX`, which matches nodes whose `upostag` value starts with `V`. This will match all verbs, including those with more specific tags like `VERB`.


In [61]:
# Tagger instantiation
verb_tagger = DepChainTagger(patterns=(v_pattern_regex,))

# Sample text with syntax layer
sample_text = "1990. aasta kuumal suvel vaatas Bureau Veritas Estline'i omanduseks saanud laeva uuesti üle."
text_obj = estnltk.Text(sample_text)
text_obj.tag_layer("morph_extended")
stanza_syntax_tagger.tag(text_obj)

# Run the tagger
verb_tagger.tag(text_obj);

In [62]:
display(text_obj["dep_chains"])
text_obj["dep_chains"].display()

RelationLayer(name='dep_chains', span_names=('starts_with_v',), attributes=('pattern_name', 'matched_text'), relations=[Relation([NamedSpan(starts_with_v: 'vaatas')], [{'pattern_name': 'regex_verb_pattern', 'matched_text': 'vaatas'}])])

1990. aasta kuumal suvel vaatas starts_with_v(0) Bureau Veritas Estline'i omanduseks saanud laeva uuesti üle.

Since all verbs in the syntax layer have `upostag` values that start with `V`, we get the same matches as with `EXACT` mode.


# TODO


### NodeConstraints


Now, that we understand the condition modes, we can define some `NodeConstraint` objects to use in our pattern. `NodeConstraint` objects specify conditions on the nodes in the pattern. They can check for specific attribute values, the presence of certain attributes, or even more complex conditions using custom functions.


Let's create a simple `NodeConstraint` that matches any verb node. We can use the `upostag` attribute and set the condition mode to `EXACT` with the value `V` to achieve this.


In [ ]:
verb_constraint = NodeConstraint(
    role="verb",
    attribute_conditions={
        "upostag": ValueCondition(mode=ConditionMode.EXACT, value="V")
    },
)

### EdgeConstraints
